In [1]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from subprocess import run, PIPE
import sys
import os
import gc
import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [2]:
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from matplotlib import pyplot as plt # Standard Plotting library
from matplotlib import cm
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
from matplotlib.ticker import (MultipleLocator, FormatStrFormatter, AutoMinorLocator)
from cartopy import crs as ccrs # Cartography library
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import matplotlib.tri as tri

In [3]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 4
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

Number of CPUs: 64, number of threads: 16, number of workers: 4, processes: False


In [4]:
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jwille//proxy/33451/status,
Dashboard: /user/jwille//proxy/33451/status,Workers: 4
Total threads: 64,Total memory: 3.64 TiB
Status: running,Using processes: False
Comm: inproc://129.132.32.66/257552/1,Workers: 4
Dashboard: /user/jwille//proxy/33451/status,Total threads: 64
Started: Just now,Total memory: 3.64 TiB
Comm: inproc://129.132.32.66/257552/4,Total threads: 16
Dashboard: /user/jwille//proxy/40697/status,Memory: 0.91 TiB
Nanny: None,


In [5]:
#works for era5

def longest_run_zeros(data_array,model):
    """
    Calculate the longest consecutive sequence of 1s along 'time' for each spatial location.

    Works for inputs with dims:
      - ('time', 'lat', 'lon')

    Returns
    -------
    xarray.DataArray
        If input was ('time','lat','lon') → dims=('lat','lon')
    """

    # pull out the raw numpy array and figure out shape
    arr = data_array.values


    # (time, lat, lon) case → flatten lat/lon into one dimension
    ntime, nlat, nlon = arr.shape
    flat = arr.reshape(ntime, nlat * nlon)
    out_dims = ['lat', 'lon']
    out_coords = {
        'lat': data_array.coords['lat'],
        'lon': data_array.coords['lon'],
    }
    grid_shape = (nlat, nlon)


    # now flat has shape (time, nloc) where nloc = ncell or nlat*nlon
    nloc = flat.shape[1]
    max_runs = np.zeros(nloc, dtype=np.int32)
    current  = np.zeros(nloc, dtype=np.int32)

    for t in range(ntime):
        # increment where 0, reset where 1
        current = np.where(flat[t] == 0, current + 1, 0)
        max_runs = np.maximum(max_runs, current)

    # if we flattened, reshape back to grid
    if grid_shape is not None:
        max_runs = max_runs.reshape(grid_shape)

    # build the output DataArray
    result = xr.DataArray(
        max_runs,
        dims=out_dims,
        coords=out_coords
    )

    return result


In [23]:
# List to store results for each year
model = 'ICON_cycle4'

results = []
latitude = [30,60]

# List of years from 2020 to 2049. Adjust accordingly.
years = range(2020, 2050)
#years = range(1990, 2020)
months = [5,6,7,8,9,10]
mask = '1'

In [24]:
for year in years:
    print(f"Processing year {year}...")
    # Open the yearly data file. Adjust the file path if needed.
    if model == 'ICON_cycle4':
        data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/2020-2050/daily_precip/cesm2_grid')
        files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_cesm2_interp_higher_ICON_ngc4008_hourly_precip_??_'+str(year)+'.nc')])[:]
        ds = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
        ds = ds.where(ds.time.dt.month.isin(months), drop=True)
        indexer = ((ds.lat>=latitude[0]) & (ds.lat<=latitude[1]))
        ds = ds.where(indexer,drop=True)
        ds = ds.load()
        
    if model == 'ICON_cycle3':
        data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom10/daily_precip/cesm2_grid')
        files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_cesm2_interp_higher_ICON_ngc3028_hourly_precip_??_'+str(year)+'.nc')])[:]
        ds = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
        ds = ds.where(ds.time.dt.month.isin(months), drop=True)
        indexer = ((ds.lat>=latitude[0]) & (ds.lat<=latitude[1]))
        ds = ds.where(indexer,drop=True)
        ds = ds.load()
        
    if model == 'IFS_cycle3':
        data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_4.4-FESOM_5/daily_precip/cesm2_grid')
        files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_cesm2_interp_IFS_4.4-FESOM_5_precip_cycle3_??_'+str(year)+'.nc')])[:]
        ds = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
        ds = ds.where(ds.time.dt.month.isin(months), drop=True)
        indexer = ((ds.lat>=latitude[0]) & (ds.lat<=latitude[1]))
        ds = ds.where(indexer,drop=True)
        ds = ds.load()


    if model == 'IFS_cycle4_historical':
        data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/1990-2020/daily_precip/cesm2_grid')
        files = sorted([str(f) for f in data_path.glob('mask_1_cesm2_interp_IFS_9-FESOM_5_hourly_precip_??_'+str(year)+'.nc')])[:]
        ds = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
        ds = ds.where(ds.time.dt.month.isin(months), drop=True)
        indexer = ((ds.lat>=latitude[0]) & (ds.lat<=latitude[1]))
        ds = ds.where(indexer,drop=True)
        ds = ds.load()

    if model == 'IFS_cycle4_future':
        data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/2020-2050/daily_precip/cesm2_grid')
        files = sorted([str(f) for f in data_path.glob('mask_1_cesm2_interp_IFS_9-FESOM_5_hourly_precip_??_'+str(year)+'.nc')])[:]
        ds = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
        ds = ds.where(ds.time.dt.month.isin(months), drop=True)
        indexer = ((ds.lat>=latitude[0]) & (ds.lat<=latitude[1]))
        ds = ds.where(indexer,drop=True)
        ds = ds.load()

    if model == 'IFS_2.8km_on':
        data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_on/daily_precip/cesm2_grid')
        files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_cesm2_interp_IFS_2.8-FESOM_5_deep_on_hourly_precip_??_'+str(year)+'.nc')])[:]
        ds = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
        ds = ds.where(ds.time.dt.month.isin(months), drop=True)
        indexer = ((ds.lat>=latitude[0]) & (ds.lat<=latitude[1]))
        ds = ds.where(indexer,drop=True)
        ds = ds.load()

    if model == 'IFS_2.8km_off':
        data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_off/daily_precip/cesm2_grid')
        files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_cesm2_interp_IFS_2.8-FESOM_5_deep_off_hourly_precip_??_'+str(year)+'.nc')])[:]
        ds = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
        ds = ds.where(ds.time.dt.month.isin(months), drop=True)
        indexer = ((ds.lat>=latitude[0]) & (ds.lat<=latitude[1]))
        ds = ds.where(indexer,drop=True)
        ds = ds.load()
        
    if model == 'IMERG':
        data_path = Path('/home/jwille/xenon_data/NextGEMS/IMERG/clean/global/cesm2_grid')
        files = sorted([str(f) for f in data_path.glob('mask_'+mask+'_daysum_cesm2_interp_IMERG_V07B_final_??_'+str(year)+'.nc4')])[:]
        ds = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['dry']
        ds = ds.where(ds.time.dt.month.isin(months), drop=True)
        indexer = ((ds.lat>=latitude[0]) & (ds.lat<=latitude[1]))
        ds = ds.where(indexer,drop=True)
        ds = ds.load()

    # Extract the 'dry' DataArray which has dimensions ('time', 'cell').
    #dry_da = ds["dry"]
    dry_da = ds

    # Calculate the longest dry spell (longest sequence of zeros) per cell.
    longest_dry = longest_run_zeros(dry_da,model)
    
    # Expand dims to add a "year" dimension so we can concatenate later.
    longest_dry = longest_dry.expand_dims(dim={"year": [year]})
    
    # Append the result to the list.
    results.append(longest_dry)

Processing year 2020...
Processing year 2021...
Processing year 2022...
Processing year 2023...
Processing year 2024...
Processing year 2025...
Processing year 2026...
Processing year 2027...
Processing year 2028...
Processing year 2029...
Processing year 2030...
Processing year 2031...
Processing year 2032...
Processing year 2033...
Processing year 2034...
Processing year 2035...
Processing year 2036...
Processing year 2037...
Processing year 2038...
Processing year 2039...
Processing year 2040...
Processing year 2041...
Processing year 2042...
Processing year 2043...
Processing year 2044...
Processing year 2045...
Processing year 2046...
Processing year 2047...
Processing year 2048...
Processing year 2049...


In [25]:
# Concatenate all yearly results into a single DataArray along the 'year' dimension.
combined_longest = xr.concat(results, dim="year")

In [26]:


dummy = xr.open_dataset('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/2020-2050/daily_precip/cesm2_grid/mask_1_cesm2_interp_IFS_9-FESOM_5_hourly_precip_10_2025.nc')['dry']
indexer = ((dummy.lat>=latitude[0]) & (dummy.lat<=latitude[1]))
dummy = dummy.where(indexer,drop=True)


dry_spell = xr.DataArray(
combined_longest,
dims=["year", "lat", "lon"],
coords={
    "lat": ("lat", dummy.coords["lat"].values),
    "lon": ("lon", dummy.coords["lon"].values)
}
)



In [27]:
dry_spell.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/dry_spells/'+model+'_cesm2_grid_summer_dry_spell_annual_30-60N.nc')


In [ ]:
print(dry_spell)